In [3]:
from roboflow import Roboflow
project = Roboflow(api_key="7nSeDlTmPggkhwLyUeJt").workspace("viren-dhanwani").project("tennis-ball-detection")
version = project.version(6)
dataset = version.download("yolov8")

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to tennis-ball-detection-6 in yolov8:: 100%|██████████| 1168/1168 [00:00<00:00, 6497.67it/s]


In [4]:
!yolo task=detect mode=train model=yolov8x.pt data=tennis-ball-detection-6/data.yaml epochs=150 imgsz=640 device=0 batch=32 workers=8 cache=True

Ultralytics 8.4.47 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=tennis-ball-detection-6/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=150, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8x.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True,

In [7]:
import cv2
from ultralytics import YOLO
from google.colab.patches import cv2_imshow

# -----------------------------
# Load models
# -----------------------------
pose_model = YOLO("yolov8s-pose.pt")
ball_model = YOLO("best.pt")

# -----------------------------
# Video input
# -----------------------------
cap = cv2.VideoCapture("input_sample_video.mp4")

# Output video writer setup
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
fps = cap.get(cv2.CAP_PROP_FPS)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

out = cv2.VideoWriter("output.mp4", fourcc, fps, (width, height))

# -----------------------------
# Processing loop
# -----------------------------
while cap.isOpened():

    ret, frame = cap.read()
    if not ret:
        break

    pose_frame = frame.copy()
    ball_frame = frame.copy()

    # -----------------------------
    # 1. POSE TRACKING
    # -----------------------------
    pose_results = pose_model.track(pose_frame, persist=True)

    output_frame = pose_results[0].plot()

    # -----------------------------
    # 2. BALL DETECTION
    # -----------------------------
    ball_results = ball_model.predict(ball_frame, conf=0.1)[0]

    for box in ball_results.boxes:

        x1, y1, x2, y2 = box.xyxy.tolist()[0]

        cx = int((x1 + x2) / 2)
        cy = int((y1 + y2) / 2)

        # Draw ball
        cv2.circle(output_frame, (cx, cy), 6, (0, 255, 255), -1)
        cv2.rectangle(output_frame,
                      (int(x1), int(y1)),
                      (int(x2), int(y2)),
                      (0, 255, 255), 2)

        cv2.putText(output_frame,
                    "TENNIS BALL",
                    (cx, cy - 10),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.6,
                    (0, 255, 255),
                    2)

    # -----------------------------
    # 3. INFO TEXT
    # -----------------------------
    cv2.putText(output_frame,
                "Tennis AI System: Pose + Ball Tracking",
                (10, 30),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.8,
                (0, 255, 0),
                2)

    # -----------------------------
    # SAVE FRAME TO VIDEO
    # -----------------------------
    out.write(output_frame)

# -----------------------------
# CLEANUP
# -----------------------------
cap.release()
out.release()

print("Processing complete. Video saved as output.mp4")

Streaming output truncated to the last 5000 lines.
Speed: 2.1ms preprocess, 11.6ms inference, 1.3ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 person, 9.0ms
Speed: 3.3ms preprocess, 9.0ms inference, 1.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 11.3ms
Speed: 2.4ms preprocess, 11.3ms inference, 0.6ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 person, 9.2ms
Speed: 2.3ms preprocess, 9.2ms inference, 1.5ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 11.3ms
Speed: 2.2ms preprocess, 11.3ms inference, 0.6ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 person, 8.9ms
Speed: 2.3ms preprocess, 8.9ms inference, 1.5ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 11.8ms
Speed: 2.7ms preprocess, 11.8ms inference, 0.7ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 person, 9.2ms
Speed: 2.9ms preprocess, 9.2ms inference, 1.5ms postp